# 03 — NLI Checker

## 1. Import Libraries

In [17]:
from transformers import pipeline


## 2. Load NLI Model

In [18]:

print("Loading model...")

nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768"
)

print("Model loaded!")

Loading model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded!


## 3. Understanding NLI

NLI checks whether the evidence supports, contradicts, or does not determine a claim.

In [19]:
evidence = "The James Webb Space Telescope was launched on December 25, 2021."
claim = "The telescope was launched in 2021."
result = nli(f"{evidence} </s></s> {claim}")
print(result)

[{'label': 'entailment', 'score': 0.9912475943565369}]


## 4. Test Entailment

In [20]:
evidence = "The James Webb Space Telescope was launched on December 25, 2021."
claim = "The telescope was launched in 2021."
result = nli(f"{evidence} </s></s> {claim}")
print(result)

[{'label': 'entailment', 'score': 0.9912475943565369}]


## 5. Test Contradiction

In [21]:
evidence = "The James Webb Space Telescope was launched on December 25, 2021."
claim = "The telescope was launched in 2020."
result = nli(f"{evidence} </s></s> {claim}")
print(result)

[{'label': 'contradiction', 'score': 0.9955217838287354}]


## 6. Test Neutral

In [22]:
evidence = "The James Webb Space Telescope was launched on December 25, 2021."
claim = "The telescope studies distant galaxies."
result = nli(f"{evidence} </s></s> {claim}")
print(result)

[{'label': 'neutral', 'score': 0.9937323927879333}]


## 7. Check Label Mapping

In [23]:
print(nli.model.config.id2label)

{0: 'contradiction', 1: 'entailment', 2: 'neutral'}


## 8. Create a Reusable Function

In [24]:
def check_claim(evidence, claim):
    result = nli(f"{evidence} </s></s> {claim}")
    return {
        "claim": claim,
        "evidence": evidence,
        "label": result[0]["label"],
        "score": result[0]["score"]
    }

In [25]:
evidence = "The telescope was launched on December 25, 2021."
claim = "The telescope was launched in 2021."
result = check_claim(evidence, claim)
print(result)

{'claim': 'The telescope was launched in 2021.', 'evidence': 'The telescope was launched on December 25, 2021.', 'label': 'entailment', 'score': 0.9911477565765381}


## 9. Test Multiple Claims

In [26]:
evidence = "The James Webb Space Telescope was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre."

In [27]:
claims = [
    "The telescope was launched in 2021.",
    "The telescope was launched using an Ariane 5 rocket.",
    "The telescope was launched from India."
]
for claim in claims:
    result = check_claim(evidence, claim)
    print(result)
    print()

{'claim': 'The telescope was launched in 2021.', 'evidence': 'The James Webb Space Telescope was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.', 'label': 'entailment', 'score': 0.99165278673172}

{'claim': 'The telescope was launched using an Ariane 5 rocket.', 'evidence': 'The James Webb Space Telescope was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.', 'label': 'entailment', 'score': 0.9906285405158997}

{'claim': 'The telescope was launched from India.', 'evidence': 'The James Webb Space Telescope was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.', 'label': 'contradiction', 'score': 0.9946492314338684}



## 10. Testing

In [28]:
chunks = [
    "The James Webb Space Telescope is a space telescope designed to conduct infrared astronomy.",
    "It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.",
    "The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth."
]
claims = [
    "The telescope was launched in 2021.",
    "The telescope was launched using an Ariane 5 rocket.",
    "The telescope is approximately 1.5 million kilometers from Earth.",
    "The telescope was launched from India."
]

In [29]:
pairs = [
    (chunks[1], claims[0]),
    (chunks[1], claims[1]),
    (chunks[2], claims[2]),
    (chunks[1], claims[3])
]

In [30]:
for evidence, claim in pairs:
    result = check_claim(evidence, claim)
    print("Claim:", claim)
    print("Result:", result["label"])
    print("Confidence:", result["score"])
    print()

Claim: The telescope was launched in 2021.
Result: neutral
Confidence: 0.9869085550308228

Claim: The telescope was launched using an Ariane 5 rocket.
Result: neutral
Confidence: 0.98773193359375

Claim: The telescope is approximately 1.5 million kilometers from Earth.
Result: entailment
Confidence: 0.9905954599380493

Claim: The telescope was launched from India.
Result: contradiction
Confidence: 0.9359614849090576



## 11. Observations

NLI compares a claim with retrieved evidence and predicts whether the evidence supports, contradicts, or does not determine the claim.